# Persetujuan Manusia (Human-in-the-Loop): Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari3/06-persetujuan-manusia-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (2 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/06-persetujuan-manusia-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Bab 6 - Solusi Latihan 1: Asisten Operasi yang Aman
Jalankan sel satu per satu dengan Shift+Enter

Agen operasi dengan empat tool dan aturan persetujuan berbeda untuk masing-masing:
- `baca_status_peralatan`: tidak pernah bertanya (hanya baca)
- `jadwalkan_perawatan`: approve / edit / reject
- `pesan_suku_cadang`: approve / edit / reject (aturan reviewer membatasi jumlah)
- `shutdown_darurat`: approve / reject saja (tidak pernah boleh diubah)

Setiap keputusan ditulis ke audit log yang dicetak di akhir.

Reviewer-nya adalah fungsi berskrip (`reviewer`) agar demo berjalan tanpa pengawasan. Ganti `reviewer` dengan `input()`,
seperti `tanya_manusia` di `code/04_approval_loop.py`, untuk membuatnya interaktif.

In [ ]:
# Langkah 1 - Menyiapkan model
import json
import os
from datetime import datetime

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Empat tool operasi

In [ ]:
MAKS_JUMLAH_PESANAN = 4
JEJAK_AUDIT: list[dict] = []


@tool
def baca_status_peralatan(lokasi: str) -> str:
    """Baca status peralatan utama di sebuah lokasi (hanya baca).

    Args:
        lokasi: Kode lokasi, contoh "GS-SR1"
    """
    return f"{lokasi}: separator V-SR1-101 OK, kompresor K-ML1-201 OK, pompa injeksi P-SR1-302 tekanan hisap rendah, flare OK"


@tool
def jadwalkan_perawatan(asset_id: str, tanggal: str, uraian: str) -> str:
    """Jadwalkan pekerjaan perawatan pada sebuah aset.

    Args:
        asset_id: Kode aset
        tanggal: Tanggal format YYYY-MM-DD
        uraian: Uraian pekerjaan
    """
    return f"Perawatan {asset_id} dijadwalkan pada {tanggal}: {uraian}"


@tool
def pesan_suku_cadang(asset_id: str, nama_barang: str, jumlah: int) -> str:
    """Buat permintaan pengadaan suku cadang untuk sebuah aset.

    Args:
        asset_id: Kode aset
        nama_barang: Nama suku cadang
        jumlah: Jumlah unit
    """
    return f"Dipesan {jumlah} unit {nama_barang} untuk {asset_id}"


@tool
def shutdown_darurat(lokasi: str, alasan: str) -> str:
    """Hentikan SELURUH fasilitas di sebuah lokasi segera (emergency shutdown).

    Args:
        lokasi: Kode lokasi
        alasan: Mengapa shutdown diperlukan
    """
    return f"EMERGENCY SHUTDOWN {lokasi}: {alasan}"

## Langkah 3 - Kebijakan, reviewer, dan pencatat audit
`reviewer` adalah pengganti otomatis untuk reviewer manusia. `catat_keputusan` menulis setiap keputusan ke `JEJAK_AUDIT`.

In [ ]:
KEBIJAKAN = {
    "baca_status_peralatan": False,
    "jadwalkan_perawatan": {"allowed_decisions": ["approve", "edit", "reject"]},
    "pesan_suku_cadang": {"allowed_decisions": ["approve", "edit", "reject"]},
    "shutdown_darurat": {"allowed_decisions": ["approve", "reject"]},
}


def reviewer(permintaan: dict) -> dict:
    """Pengganti otomatis untuk reviewer manusia."""
    nama, argumen = permintaan["name"], permintaan["args"]

    if nama == "pesan_suku_cadang" and argumen["jumlah"] > MAKS_JUMLAH_PESANAN:
        diperbaiki = {**argumen, "jumlah": MAKS_JUMLAH_PESANAN}
        return {"type": "edit", "edited_action": {"name": nama, "args": diperbaiki}}
    if nama == "shutdown_darurat":
        return {"type": "reject", "message": "Shutdown harus dikonfirmasi oleh manajer jaga."}
    return {"type": "approve"}


def catat_keputusan(permintaan: dict, keputusan: dict) -> None:
    JEJAK_AUDIT.append(
        {
            "waktu": datetime.now().strftime("%H:%M:%S"),
            "tool": permintaan["name"],
            "argumen": permintaan["args"],
            "keputusan": keputusan["type"],
            "catatan": keputusan.get("message") or keputusan.get("edited_action", {}).get("args"),
        }
    )

## Langkah 4 - Membuat agen dengan persetujuan

In [ ]:
agent = create_agent(
    model,
    tools=[baca_status_peralatan, jadwalkan_perawatan, pesan_suku_cadang, shutdown_darurat],
    system_prompt=(
        "Anda asisten operasi untuk gathering station. Pakai tool untuk melakukan yang diminta pengguna. "
        "Jika sebuah pemanggilan ditolak, jelaskan alasannya dan jangan coba lagi. Jawab singkat dalam Bahasa Indonesia."
    ),
    middleware=[HumanInTheLoopMiddleware(interrupt_on=KEBIJAKAN)],
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "ops-1"}}

## Langkah 5 - Satu permintaan, beberapa tindakan, banyak keputusan
Selama masih ada interrupt, setiap tindakan yang diusulkan ditinjau dan dicatat, lalu agen dilanjutkan.

In [ ]:
teks_permintaan = (
    "Periksa peralatan di GS-SR1. Lalu jadwalkan perawatan untuk peralatan yang bermasalah pada 2026-08-10, "
    "pesan 10 unit seal pompa untuk P-SR1-302, dan jika ada yang tampak berbahaya, lakukan shutdown darurat."
)
print(f"Pengguna: {teks_permintaan}\n")

result = agent.invoke({"messages": [HumanMessage(content=teks_permintaan)]}, config)

while "__interrupt__" in result:
    payload = result["__interrupt__"][0].value
    keputusan_daftar = []
    for permintaan in payload["action_requests"]:
        keputusan = reviewer(permintaan)
        catat_keputusan(permintaan, keputusan)
        print(f"Usulan {permintaan['name']}({permintaan['args']}) -> {keputusan['type'].upper()}")
        keputusan_daftar.append(keputusan)
    result = agent.invoke(Command(resume={"decisions": keputusan_daftar}), config)

print(f"\nAgen: {result['messages'][-1].content}\n")

Pengguna: Periksa peralatan di GS-SR1. Lalu jadwalkan perawatan untuk peralatan yang bermasalah pada 2026-08-10, pesan 10 unit seal pompa untuk P-SR1-302, dan jika ada yang tampak berbahaya, lakukan shutdown darurat.

Usulan jadwalkan_perawatan({'asset_id': 'P-SR1-302', 'tanggal': '2026-08-10', 'uraian': 'Perawatan pompa injeksi - tekanan hisap rendah'}) -> APPROVE
Usulan pesan_suku_cadang({'asset_id': 'P-SR1-302', 'nama_barang': 'seal pompa', 'jumlah': 10}) -> EDIT

Agen: Berikut ringkasan tindakan yang telah dilakukan:

1. **Pemeriksaan peralatan GS-SR1**:
   - Separator V-SR1-101:  OK
   - Kompresor K-ML1-201:  OK
   - Pompa injeksi P-SR1-302:  Tekanan hisap rendah
   - Flare:  OK

2. **Perawatan dijadwalkan** untuk P-SR1-302 pada **2026-08-10** dengan uraian perbaikan tekanan hisap rendah.

3. **Pengadaan suku cadang**: 4 unit **seal pompa** berhasil dipesan untuk P-SR1-302 (jumlah disesuaikan dari permintaan awal 10 unit menjadi 4 unit oleh reviewer).

4. **Shutdown darurat**: **T

## Langkah 6 - Audit log

In [ ]:
print("Audit log")
print("-" * 60)
for entri in JEJAK_AUDIT:
    print(json.dumps(entri, ensure_ascii=False))

Audit log
------------------------------------------------------------
{"waktu": "07:54:08", "tool": "jadwalkan_perawatan", "argumen": {"asset_id": "P-SR1-302", "tanggal": "2026-08-10", "uraian": "Perawatan pompa injeksi - tekanan hisap rendah"}, "keputusan": "approve", "catatan": null}
{"waktu": "07:54:08", "tool": "pesan_suku_cadang", "argumen": {"asset_id": "P-SR1-302", "nama_barang": "seal pompa", "jumlah": 10}, "keputusan": "edit", "catatan": {"asset_id": "P-SR1-302", "nama_barang": "seal pompa", "jumlah": 4}}


### Penjelasan Solusi Latihan 6.1: asisten operasi yang aman

Setiap tool punya aturan persetujuan sendiri, ditulis dalam satu kamus `KEBIJAKAN`, dan `HumanInTheLoopMiddleware` menjeda agen sebelum tool yang diatur berjalan.

- `TODO(B6.1)`: tool baca tidak pernah bertanya (`False`). Tool tulis boleh `approve`, `edit`, atau `reject`. `shutdown_darurat` hanya `approve` atau `reject`, jadi tidak ada jalan untuk mengubah argumennya.
- `TODO(B6.2)`: `reviewer` adalah pengganti reviewer manusia: jumlah pesanan di atas batas diubah (`edit`), shutdown ditolak dengan alasan, sisanya disetujui.
- `TODO(B6.3)`: middleware dipasang dengan `interrupt_on=KEBIJAKAN`, dan `checkpointer` wajib ada agar agen bisa dilanjutkan.
- `TODO(B6.4)`: `Command(resume={"decisions": [...]})` dengan `config` yang sama melanjutkan agen. Urutan keputusan harus sama dengan urutan `action_requests`.

Keluaran memperlihatkan dua usulan: perawatan P-SR1-302 disetujui, dan pesanan 10 unit seal pompa diubah menjadi 4 sebelum tool berjalan. Jawaban akhir agen pun menyebut jumlah 4. Dalam eksekusi ini agen tidak mengusulkan shutdown darurat sama sekali (ia menilai tidak ada yang berbahaya), jadi jalur penolakan shutdown tidak teruji kali ini. Audit log di akhir mencatat setiap keputusan beserta argumen yang benar-benar dijalankan.

## Bab 6 - Solusi Latihan Bonus: Persetujuan yang Bertahan Lebih Lama dari Program
Jalankan sel satu per satu dengan Shift+Enter

Di sistem nyata, manusia jarang duduk di depan terminal. Seorang manajer mungkin menyetujui permintaan
satu jam kemudian, dari proses yang berbeda. Solusi ini membagi pekerjaan:

- `submit`: jalankan agen sampai perlu persetujuan, lalu KELUAR. Permintaan yang menunggu disimpan oleh
  checkpointer (`SqliteSaver`) dan dicatat dalam file JSON kecil.
- `decide`: proses yang sepenuhnya terpisah memuat thread tersimpan dan melanjutkannya.

Karena agen yang berhenti tersimpan di SQLite (bab 5), kedua langkah bisa berjalan berjam-jam berselang.

Demo di bawah memakai folder sementara dan membuat objek agen BARU untuk setiap langkah. Fungsi `submit`,
`pending`, dan `decide` juga bisa dipanggil sendiri-sendiri (lihat sel opsional di akhir) dengan
file tetap yang bertahan setelah program selesai.

In [ ]:
# Langkah 1 - Menyiapkan model dan alat bantu
import json
import os
import sqlite3
import tempfile
import uuid

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command

from migas_shared.config import ROOT

load_dotenv()

True

## Langkah 2 - Tool dan pembuat agen
`buat_agen` membuat objek agen BARU setiap kali: seluruh memori berasal dari file SQLite.

In [ ]:
@tool
def pesan_suku_cadang(asset_id: str, nama_barang: str, jumlah: int) -> str:
    """Buat permintaan pengadaan suku cadang untuk sebuah aset.

    Args:
        asset_id: Kode aset
        nama_barang: Nama suku cadang
        jumlah: Jumlah unit
    """
    return f"Dipesan {jumlah} unit {nama_barang} untuk {asset_id}"


def buat_agen(db_path: str):
    """Objek agen BARU setiap kali: seluruh memori berasal dari file SQLite."""
    model = ChatOpenAI(
        model=os.getenv("ONLINE_MODEL"),
        base_url=os.getenv("ONLINE_BASE_URL"),
        api_key=os.getenv("ONLINE_API_KEY"),
        reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    )
    connection = sqlite3.connect(db_path, check_same_thread=False)
    agent = create_agent(
        model,
        tools=[pesan_suku_cadang],
        system_prompt="Anda menangani permintaan suku cadang. Pakai tool. Jawab singkat dalam Bahasa Indonesia.",
        middleware=[HumanInTheLoopMiddleware(interrupt_on={"pesan_suku_cadang": True})],
        checkpointer=SqliteSaver(connection),
    )
    return agent, connection

## Langkah 3 - Antrean persetujuan: submit, pending, decide
Antrean disimpan sebagai file JSON. `submit` menaruh permintaan yang berhenti ke antrean, `pending` menampilkannya,
dan `decide` melanjutkan thread tersimpan dengan sebuah keputusan.

In [ ]:
def muat_antrean(path: str) -> dict:
    if not os.path.exists(path):
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def simpan_antrean(path: str, antrean: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(antrean, f, indent=2, ensure_ascii=False)


def submit(db_path: str, antrean_path: str, teks: str) -> str:
    """Jalankan agen. Bila berhenti, simpan permintaan ke antrean dan kembalikan id thread."""
    thread_id = uuid.uuid4().hex[:8]
    agent, connection = buat_agen(db_path)
    config = {"configurable": {"thread_id": thread_id}}

    result = agent.invoke({"messages": [HumanMessage(content=teks)]}, config)
    connection.close()

    if "__interrupt__" not in result:
        print(f"Tidak perlu persetujuan. Agen berkata: {result['messages'][-1].content}")
        return thread_id

    aksi = result["__interrupt__"][0].value["action_requests"][0]
    antrean = muat_antrean(antrean_path)
    antrean[thread_id] = {"tool": aksi["name"], "args": aksi["args"], "permintaan": teks}
    simpan_antrean(antrean_path, antrean)
    print(f"Permintaan {thread_id} menunggu persetujuan: {aksi['name']}({aksi['args']})")
    return thread_id


def pending(antrean_path: str) -> None:
    antrean = muat_antrean(antrean_path)
    if not antrean:
        print("Tidak ada yang menunggu persetujuan")
    for thread_id, item in antrean.items():
        print(f"Menunggu {thread_id}: {item['tool']}({item['args']})")


def decide(db_path: str, antrean_path: str, thread_id: str, putusan: str, alasan: str = "") -> None:
    """'Sesi' terpisah memuat thread tersimpan dan melanjutkannya dengan sebuah keputusan."""
    antrean = muat_antrean(antrean_path)
    if thread_id not in antrean:
        print(f"Tidak ada permintaan tertunda dengan id {thread_id}")
        return

    keputusan = (
        {"type": "approve"}
        if putusan == "approve"
        else {"type": "reject", "message": alasan or "Ditolak oleh pemberi persetujuan"}
    )
    agent, connection = buat_agen(db_path)  # agen baru, tidak ada apa pun di RAM
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(Command(resume={"decisions": [keputusan]}), config)
    connection.close()

    del antrean[thread_id]
    simpan_antrean(antrean_path, antrean)
    label = "disetujui" if putusan == "approve" else "ditolak"
    print(f"{thread_id} {label} -> {result['messages'][-1].content}")

## Langkah 4 - Folder sementara untuk demo
Database dan antrean demo berada di folder sementara agar tidak meninggalkan file.

In [ ]:
folder_demo = tempfile.TemporaryDirectory(ignore_cleanup_errors=True)
db_path = os.path.join(folder_demo.name, "persetujuan.db")
antrean_path = os.path.join(folder_demo.name, "antrean.json")

## Langkah 5 - Proses A: operator mengirim dua permintaan, lalu program KELUAR
Setiap `submit` membuat objek agen baru dan berhenti di interrupt persetujuan. Permintaan pertama wajar.

In [ ]:
print("Proses A: operator mengirim dua permintaan, lalu program KELUAR")
id1 = submit(db_path, antrean_path, "Pesan 2 unit modul VSD FD500 untuk ESP SR-020")

Proses A: operator mengirim dua permintaan, lalu program KELUAR
Permintaan a17f9e3b menunggu persetujuan: pesan_suku_cadang({'asset_id': 'ESP SR-020', 'nama_barang': 'modul VSD FD500', 'jumlah': 2})


## Langkah 6 - Proses A: permintaan kedua (jumlah besar)

In [ ]:
id2 = submit(db_path, antrean_path, "Pesan 40 unit gasket flange untuk K-ML1-201")

Permintaan 0443d2bf menunggu persetujuan: pesan_suku_cadang({'asset_id': 'K-ML1-201', 'nama_barang': 'gasket flange', 'jumlah': 40})


## Langkah 7 - Proses B: manajer melihat antrean

In [ ]:
print("Proses B (nanti): manajer melihat antrean")
pending(antrean_path)

Proses B (nanti): manajer melihat antrean
Menunggu a17f9e3b: pesan_suku_cadang({'asset_id': 'ESP SR-020', 'nama_barang': 'modul VSD FD500', 'jumlah': 2})
Menunggu 0443d2bf: pesan_suku_cadang({'asset_id': 'K-ML1-201', 'nama_barang': 'gasket flange', 'jumlah': 40})


## Langkah 8 - Proses B: manajer memutuskan
Keputusan dijalankan dengan objek agen yang benar-benar baru. Permintaan pertama disetujui, kedua ditolak dengan alasan.

In [ ]:
print("Proses B: manajer memutuskan, memakai objek agen yang benar-benar baru")
decide(db_path, antrean_path, id1, "approve")
decide(db_path, antrean_path, id2, "reject", "Stok gasket di gudang Kenanga masih cukup untuk 3 bulan.")

Proses B: manajer memutuskan, memakai objek agen yang benar-benar baru
a17f9e3b disetujui -> Permintaan pengadaan suku cadang berhasil diproses:

- **Aset:** ESP SR-020
- **Barang:** Modul VSD FD500
- **Jumlah:** 2 unit

Pesanan telah diajukan. Apakah ada lagi yang bisa saya bantu?
0443d2bf ditolak -> Permintaan pengadaan **40 unit gasket flange** untuk aset **K-ML1-201** **ditolak** karena stok gasket di gudang Kenanga masih cukup untuk kebutuhan 3 bulan ke depan.

Apakah ada suku cadang lain yang perlu dipesan?


## Langkah 9 - Antrean sesudahnya

In [ ]:
print("Antrean sesudahnya")
pending(antrean_path)
folder_demo.cleanup()

Antrean sesudahnya
Tidak ada yang menunggu persetujuan


## Langkah 10 - Memakai file tetap (opsional)
Untuk mencoba pemisahan proses yang sebenarnya, panggil fungsi di atas dengan file yang bertahan di disk.
Hapus tanda `#` pada tiga baris di bawah, jalankan `submit`, ulangi sel di sesi lain (atau mulai ulang kernel),
lalu jalankan `pending` dan `decide`.

In [ ]:
DB_BAWAAN = str(ROOT / "hari3" / "06-persetujuan-manusia" / "persetujuan.db")
ANTREAN_BAWAAN = str(ROOT / "hari3" / "06-persetujuan-manusia" / "antrean_persetujuan.json")

# submit(DB_BAWAAN, ANTREAN_BAWAAN, "Pesan 2 unit modul VSD FD500 untuk ESP SR-020")
# pending(ANTREAN_BAWAAN)
# decide(DB_BAWAAN, ANTREAN_BAWAAN, "<thread_id>", "approve")

### Penjelasan Solusi Latihan 6.2: persetujuan yang bertahan lebih lama dari program

Persetujuan tidak harus terjadi saat program yang sama masih hidup. Keadaan agen yang berhenti disimpan di SQLite, jadi manajer bisa memutuskan satu jam kemudian dari proses lain.

- `TODO(B6.5)`: `buat_agen` dengan `SqliteSaver` dan `interrupt_on={"pesan_suku_cadang": True}`. Ia membuat objek agen baru setiap dipanggil; seluruh ingatan berasal dari file.
- `TODO(B6.6)`: aksi yang menunggu dibaca dari `result["__interrupt__"][0].value["action_requests"]`.
- `TODO(B6.7)`: permintaan ditulis ke antrean JSON bersama `thread_id`-nya.
- `TODO(B6.8)`: `decide` membuat agen baru, lalu melanjutkan thread tersimpan dengan `Command(resume=...)`.

Keluaran memperlihatkan alurnya: dua permintaan masuk antrean, `pending` menampilkan keduanya, lalu satu disetujui dan satu ditolak dengan alasan stok masih cukup, dan antrean kosong sesudahnya. Perhatikan argumen pada permintaan pertama: `asset_id` terisi "ESP SR-020", padahal kode asetnya `SR-020`. Model salah mengurai, dan reviewer manusia melihatnya sebelum tool berjalan. Itulah gunanya tahap persetujuan: ia menangkap kesalahan yang tidak terlihat di tingkat prompt.